# 07 — Lung-crop preprocessing and shortcut probes (reviewer Q1)

**Purpose.** Remove the image regions that carry acquisition shortcuts (borders, text markers, collimation,
abdomen) by cropping every radiograph to the bounding box of its lung fields, then check how much class
information the cropped images still leak through low-level cues. Notebook 02 trains on these crops
(`opts lungcrop=True`), so attention effects can be re-estimated under the constraint.

**Steps**
1. Lung masks for all 7,135 original images with the TorchXRayVision PSPNet anatomy model (as in notebook 05).
2. Bounding box of the union of both lungs, expanded by 8% per side (so the later 224/256 crop cannot clip lung), in normalised coordinates → `lung_boxes.csv`.
   Images whose mask fails QC (lung area < 8% or > 70% of the frame) keep the full frame; the count is reported.
3. QC grid of crops (inspect before training).
4. Shortcut probes on full images vs lung crops: (a) logistic regression on 32×32 thumbnails, (b) the
   border-only probe of notebook 01 (central 60% blanked). A drop on crops quantifies the shortcut signal removed.
   File-metadata shortcuts (format, size) are not visible to the network after decoding and resizing, so they are
   not re-tested; JPEG/PNG compression texture inside the lungs can survive cropping and is a stated limitation.

**Turn the GPU on** (Settings → Accelerator → GPU T4 x2 or P100) **and Internet on.** GPU ≈ 10–20 min for 7,135 images.
On CPU the segmentation takes > 12 h and Kaggle kills the session, so the notebook stops immediately without a GPU.
Progress is saved to `lung_boxes_partial.csv` every 800 images and the loop stops itself before Kaggle's time
limit, so a run that does not finish still leaves output: attach it to the next version and it resumes.
Output: `/kaggle/working/lungcrop/`.

In [ ]:
import os, glob, json
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
try:
    import torchxrayvision as xrv
except ImportError:
    os.system('pip install -q torchxrayvision'); import torchxrayvision as xrv
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score

ORIGINAL_ROOT = '/kaggle/input/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis'
OUT = '/kaggle/working/lungcrop'; os.makedirs(OUT, exist_ok=True)
CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
MARGIN = 0.08          # box expansion per side (fraction of box size); ≥0.072 so the 224/256 centre crop keeps the whole box
QC_LUNG_FRAC = (0.08, 0.70)
BATCH = 16
MAX_HOURS = 10.5      # stop the mask loop and save before Kaggle's 12 h limit
ALLOW_CPU = False     # leave False: CPU takes > 12 h
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda' and not ALLOW_CPU:
    raise RuntimeError('No GPU. In Kaggle: Settings → Accelerator → GPU (T4 or P100), then run again. '
                       'CPU segmentation of 7,135 images exceeds the 12 h session limit.')
import time; T0 = time.time()

def find_root(preferred):
    ok = lambda r: all(os.path.isdir(os.path.join(r, 'test', c)) for c in CLASS_NAMES)
    if ok(preferred): return preferred
    c = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/test/TURBERCULOSIS', recursive=True)})
    c = [x for x in c if ok(x)]
    if not c: raise FileNotFoundError('Attach the jtiptj chest X-ray dataset')
    return max(c, key=lambda x: len(os.listdir(os.path.join(x, 'train', 'PNEUMONIA'))))
ORIGINAL_ROOT = find_root(ORIGINAL_ROOT); print('dataset root:', ORIGINAL_ROOT, '| device', device)

items = []
for sp in ['train', 'val', 'test']:
    for ci, c in enumerate(CLASS_NAMES):
        d = os.path.join(ORIGINAL_ROOT, sp, c)
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(IMG_EXTS): items.append((sp, ci, os.path.join(d, fn)))
print(len(items), 'images')

In [ ]:
# ── 1–2. lung masks → boxes ───────────────────────────────────────────────────
seg = xrv.baseline_models.chestx_det.PSPNet().to(device).eval()
iL, iR = seg.targets.index('Left Lung'), seg.targets.index('Right Lung')
rows, done_rel = [], set()
for f in glob.glob('/kaggle/input/**/lungcrop/lung_boxes_partial.csv', recursive=True) + \
         glob.glob('/kaggle/input/**/lungcrop/lung_boxes.csv', recursive=True):
    prev = pd.read_csv(f); rows = prev.to_dict('records'); done_rel = set(prev.relpath)
    print(f'resuming from {f}: {len(done_rel)} images already segmented'); break
todo = [it for it in items if os.path.relpath(it[2], ORIGINAL_ROOT) not in done_rel]
finished = True
with torch.no_grad():
    for b0 in range(0, len(todo), BATCH):
        if (time.time() - T0) / 3600 > MAX_HOURS:
            finished = False; print(f'time budget reached at {len(rows)}/{len(items)} images'); break
        batch = todo[b0:b0 + BATCH]
        x = np.stack([np.asarray(Image.open(p).convert('RGB').convert('L').resize((512, 512), Image.BILINEAR), np.float32)
                      for _, _, p in batch])
        x = torch.from_numpy(xrv.datasets.normalize(x, 255))[:, None].to(device)
        m = (torch.sigmoid(seg(x)) > 0.5)[:, [iL, iR]].any(1).cpu().numpy()      # (B, 512, 512) union of lungs
        for (sp, ci, p), mk in zip(batch, m):
            frac = float(mk.mean()); ys, xs = np.where(mk)
            ok = QC_LUNG_FRAC[0] <= frac <= QC_LUNG_FRAC[1] and len(xs) > 0
            if ok:
                x0, x1, y0, y1 = xs.min() / 512, (xs.max() + 1) / 512, ys.min() / 512, (ys.max() + 1) / 512
                mw, mh = MARGIN * (x1 - x0), MARGIN * (y1 - y0)
                x0, x1, y0, y1 = max(0, x0 - mw), min(1, x1 + mw), max(0, y0 - mh), min(1, y1 + mh)
            else:
                x0, y0, x1, y1 = 0.0, 0.0, 1.0, 1.0
            rows.append(dict(relpath=os.path.relpath(p, ORIGINAL_ROOT), split=sp, cls=CLASS_NAMES[ci], lung_frac=frac,
                             qc_ok=ok, x0=x0, y0=y0, x1=x1, y1=y1))
        if b0 % 800 == 0:
            pd.DataFrame(rows).to_csv(f'{OUT}/lung_boxes_partial.csv', index=False)
            print(f'  {len(rows)}/{len(items)}  ({(time.time() - T0) / 60:.1f} min)')
B = pd.DataFrame(rows); B['box_area'] = (B.x1 - B.x0) * (B.y1 - B.y0)
if not finished or len(B) < len(items):
    B.to_csv(f'{OUT}/lung_boxes_partial.csv', index=False)
    raise SystemExit(f'Partial: {len(B)}/{len(items)} images. Attach this version\'s output and run again to resume.')
B = B.set_index('relpath').loc[[os.path.relpath(p, ORIGINAL_ROOT) for _, _, p in items]].reset_index()   # original order
B.to_csv(f'{OUT}/lung_boxes.csv', index=False)
if os.path.exists(f'{OUT}/lung_boxes_partial.csv'): os.remove(f'{OUT}/lung_boxes_partial.csv')
print(f'segmentation done in {(time.time() - T0) / 60:.1f} min')
print('QC pass by split/class:'); display(B.groupby(['split', 'cls']).qc_ok.agg(['sum', 'size']))
print('median box area (fraction of frame) by class:'); display(B[B.qc_ok].groupby('cls').box_area.median().round(3))

In [ ]:
# ── 3. QC grid ────────────────────────────────────────────────────────────────
rng = np.random.RandomState(0)
pick = [rng.choice(B.index[(B.cls == c)]) for c in CLASS_NAMES for _ in range(3)]
fig, ax = plt.subplots(len(CLASS_NAMES), 6, figsize=(15, 2.6 * len(CLASS_NAMES)))
for k, i in enumerate(pick):
    r = B.loc[i]; im = Image.open(os.path.join(ORIGINAL_ROOT, r.relpath)).convert('L'); W, H = im.size
    a = ax[k // 3, 2 * (k % 3)]; a.imshow(im, cmap='gray')
    a.add_patch(plt.Rectangle((r.x0 * W, r.y0 * H), (r.x1 - r.x0) * W, (r.y1 - r.y0) * H, fill=False, ec='lime', lw=1.5))
    a.set_title(f"{r.cls} lung={r.lung_frac:.2f} {'OK' if r.qc_ok else 'FULL FRAME'}", fontsize=7); a.axis('off')
    b = ax[k // 3, 2 * (k % 3) + 1]
    b.imshow(im.crop((int(r.x0 * W), int(r.y0 * H), int(r.x1 * W), int(r.y1 * H))).resize((224, 224)), cmap='gray')
    b.set_title('lung crop', fontsize=7); b.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT}/lungcrop_qc.png', dpi=120); plt.show()

In [ ]:
# ── 4. shortcut probes: full image vs lung crop ───────────────────────────────
def thumb(p, box=None, s=32):
    im = Image.open(p).convert('RGB').convert('L')
    if box is not None:
        W, H = im.size; im = im.crop((int(box[0] * W), int(box[1] * H), int(box[2] * W), int(box[3] * H)))
    return np.asarray(im.resize((s, s), Image.BILINEAR), np.float32) / 255.

def border_only(t, keep=0.2):
    t = t.copy(); b = int(t.shape[0] * keep); t[b:-b, b:-b] = 0; return t

paths = [os.path.join(ORIGINAL_ROOT, r) for r in B.relpath]; y = B.cls.map({c: i for i, c in enumerate(CLASS_NAMES)}).values
tr, te = (B.split == 'train').values, (B.split == 'test').values
full = np.stack([thumb(p) for p in paths]); crop = np.stack([thumb(p, (r.x0, r.y0, r.x1, r.y1)) for p, r in zip(paths, B.itertuples())])

def probe(X):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5, class_weight='balanced'))
    clf.fit(X[tr].reshape(tr.sum(), -1), y[tr]); p = clf.predict(X[te].reshape(te.sum(), -1))
    return accuracy_score(y[te], p), f1_score(y[te], p, average='macro')
res = []
for name, X in [('32×32 thumbnail — full image', full), ('32×32 thumbnail — lung crop', crop),
                ('border only (central 60% blanked) — full image', np.stack([border_only(t) for t in full])),
                ('border only (central 60% blanked) — lung crop', np.stack([border_only(t) for t in crop]))]:
    a, f = probe(X); res.append(dict(probe=name, test_acc=a, test_macro_f1=f))
P = pd.DataFrame(res); P.to_csv(f'{OUT}/shortcut_probes_lungcrop.csv', index=False); display(P.round(4))
json.dump(dict(n_images=len(B), n_qc_fail=int((~B.qc_ok).sum()), margin=MARGIN, qc_rule=QC_LUNG_FRAC,
               probes=dict(zip(P.probe, P.test_macro_f1.round(4)))), open(f'{OUT}/lungcrop_summary.json', 'w'), indent=1)
print('Attach this notebook\'s output to notebook 02 (REV1 lung-crop runs) and to notebook 03.')